# Analyze Evaluation Results

Unblinds the labeled sample and reports precision per source cell and per stratum.

Run this **after** you have filled in the `label` column of `evaluation_sample.csv`.


In [1]:
import pandas as pd

labels = pd.read_csv("evaluation_sample.csv")
key = pd.read_csv("evaluation_sample_key.csv")

df = labels.merge(key, on="pair_id", how="left")
print(f"Total rows: {len(df):,}")

# Normalize empty/missing labels
df["label"] = df["label"].fillna("").astype(str).str.strip().str.lower()

print()
print("Label distribution:")
print(df["label"].value_counts(dropna=False).to_string())

# Sanity: warn if any rows are unlabeled or have invalid labels
valid = {"match", "no_match", "unclear"}
unlabeled = df[~df.label.isin(valid)]
if len(unlabeled):
    print(f"\n⚠ {len(unlabeled)} rows are unlabeled or have invalid labels:")
    print(unlabeled["label"].value_counts(dropna=False).to_string())


Total rows: 250

Label distribution:
label
match       242
no_match      4
unclear       4


## Precision by source cell and stratum

In [2]:
def precision_table(group_col):
    out = (df.groupby(group_col)["label"]
             .value_counts()
             .unstack(fill_value=0))
    for c in ["match", "no_match", "unclear"]:
        if c not in out.columns:
            out[c] = 0
    out["n_labeled"]    = out["match"] + out["no_match"]
    out["precision"]    = out["match"] / out["n_labeled"].replace(0, pd.NA)
    out["unclear_rate"] = out["unclear"] / (out["match"] + out["no_match"] + out["unclear"])
    return out[["match", "no_match", "unclear", "n_labeled", "precision", "unclear_rate"]]

print("=" * 60)
print("PRECISION BY SOURCE CELL")
print("=" * 60)
print(precision_table("source_cell").round(3).to_string())
print()
print("=" * 60)
print("PRECISION BY STRATUM")
print("=" * 60)
print(precision_table("stratum").round(3).to_string())


PRECISION BY SOURCE CELL
label         match  no_match  unclear  n_labeled  precision  unclear_rate
source_cell                                                               
agreement        50         0        0         50      1.000          0.00
cluster_only     99         0        1         99      1.000          0.01
fuzzy_only       93         4        3         97      0.959          0.03

PRECISION BY STRATUM
label             match  no_match  unclear  n_labeled  precision  unclear_rate
stratum                                                                       
agreement            50         0        0         50      1.000          0.00
cluster_size_gt5     50         0        0         50      1.000          0.00
cluster_size_le5     49         0        1         49      1.000          0.02
score_82_90          45         4        1         49      0.918          0.02
score_90_100         48         0        2         48      1.000          0.04


## Confidence intervals

Wilson 95% CI on the precision estimates — gives a sense of how much
your sample size limits certainty.


In [3]:
from math import sqrt

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (center - half, center + half)

print(f"{'cell':20s} {'precision':>10s} {'95% CI':>20s}  n")
for cell in ["agreement", "fuzzy_only", "cluster_only"]:
    sub = df[df.source_cell == cell]
    k = (sub.label == "match").sum()
    n = ((sub.label == "match") | (sub.label == "no_match")).sum()
    p = k / n if n else float('nan')
    lo, hi = wilson_ci(k, n)
    print(f"{cell:20s} {p:>10.1%}   [{lo:.1%}, {hi:.1%}]   n={n}")


cell                  precision               95% CI  n
agreement                100.0%   [92.9%, 100.0%]   n=50
fuzzy_only                95.9%   [89.9%, 98.4%]   n=97
cluster_only             100.0%   [96.3%, 100.0%]   n=99


## False positives (no_match labels) by source cell

In [4]:
errors = df[df.label == "no_match"]
print(f"Total no_match labels: {len(errors):,}\n")

for cell in ["agreement", "fuzzy_only", "cluster_only"]:
    cell_errors = errors[errors.source_cell == cell]
    print("=" * 70)
    print(f"{cell.upper()} — {len(cell_errors)} no_match (showing up to 10)")
    print("=" * 70)
    for _, row in cell_errors.head(10).iterrows():
        print(f"  R: {row.r_company_name!r:60s} ({row.r_city}, {row.r_state})")
        print(f"  C: {row.c_company_name!r:60s} ({row.c_city}, {row.c_state})")
        if pd.notna(row.notes) and str(row.notes).strip():
            print(f"    notes: {row.notes}")
        print()


Total no_match labels: 4

AGREEMENT — 0 no_match (showing up to 10)
FUZZY_ONLY — 4 no_match (showing up to 10)
  R: 'CROWN NURSING HOME'                                         (BROOKLYN, NY)
  C: 'Carlton Nursing Home'                                       (BROOKLYN, NY)

  R: 'SANTA FE HOTEL & CASINO'                                    (LAS VEGAS, NV)
  C: 'SAHARA HOTEL AND CASINO'                                    (LAS VEGAS, NV)

  R: 'SANTA FE HOTEL & CASINO'                                    (LAS VEGAS, NV)
  C: 'SANDS HOTEL AND CASINO'                                     (LAS VEGAS, NV)

  R: 'WASHINGTON HILTON HOTEL'                                    (WASHINGTON, DC)
  C: 'CROWN HOTELS (WASHINGTON) INC'                              (WASHINGTON, DC)

CLUSTER_ONLY — 0 no_match (showing up to 10)


## Unclear labels by source cell

In [5]:
unclear = df[df.label == "unclear"]
print(f"Total unclear labels: {len(unclear):,}\n")

print(unclear.groupby("source_cell").size().to_string())

if len(unclear):
    print("\nSample of unclear cases (up to 10):")
    for _, row in unclear.head(10).iterrows():
        print(f"  [{row.source_cell}]  R: {row.r_company_name!r}  /  C: {row.c_company_name!r}")
        if pd.notna(row.notes) and str(row.notes).strip():
            print(f"    notes: {row.notes}")


Total unclear labels: 4

source_cell
cluster_only    1
fuzzy_only      3

Sample of unclear cases (up to 10):
  [cluster_only]  R: 'Sunglass Products, Inc. d/b/a '  /  C: 'Personal Optics AKA Style Eyes of California Vision, Sun Glass'
  [fuzzy_only]  R: 'Pet Co., Industries           '  /  C: 'Petco Industries'
  [fuzzy_only]  R: 'Concrete Walls Inc.           '  /  C: 'Concrete Form Walls'
  [fuzzy_only]  R: 'TAPCO SOUTH'  /  C: 'TAP SOUTH, INC.'


## Headline summary

In [6]:
prec = precision_table("source_cell")
n_total    = len(df)
n_labeled  = ((df.label == "match") | (df.label == "no_match")).sum()
n_unclear  = (df.label == "unclear").sum()
n_blank    = (~df.label.isin(["match", "no_match", "unclear"])).sum()

print(f"Sampled:        {n_total:,}")
print(f"Labeled (M/NM): {n_labeled:,}")
print(f"Unclear:        {n_unclear:,}")
print(f"Unlabeled:      {n_blank:,}")
print()
print("Precision by method:")
for cell in ["agreement", "fuzzy_only", "cluster_only"]:
    if cell in prec.index:
        p = prec.loc[cell, "precision"]
        n = int(prec.loc[cell, "n_labeled"])
        print(f"  {cell:15s}  {p:.1%}  (n={n})")


Sampled:        250
Labeled (M/NM): 246
Unclear:        4
Unlabeled:      0

Precision by method:
  agreement        100.0%  (n=50)
  fuzzy_only       95.9%  (n=97)
  cluster_only     100.0%  (n=99)
